# 04 — Prüfmaße für die Zahndicke (DIN 21773)

**Ziel.** Dieses Notebook dokumentiert die Prüfmaße, mit denen die Zahndicke eines außenverzahnten Zylinderrads gemessen wird: Zahnweite, Kugel- und Rollenmaße, Zahndickensehne — jeweils als Nennmaß und als oberes, mittleres und unteres Grenzmaß — und den umgekehrten Weg: wie ein Prüfmaß einer Eingabe die Profilverschiebung bestimmt.

**Regeln.** Das Notebook importiert `gearcore` und rechnet nichts selbst (ADR-104). Jede Gleichung ist eine Funktion mit Normverweis (`equations_of`). Zahlen aus STplus sind Orakelwerte, keine Normwerte.

| Quelle | Dokument | verwendet für |
|---|---|---|
| `DIN21773:2014` | DIN 21773:2014-08 | §4 Nenn- und Grenzmaße, §5/§6 Zahndickensehne, §7 Zahnweite, §8–§11 Kugel- und Rollenmaße, §13 überschnittener Kopfkreis, §14 Abmaße |
| `DIN3977:1981` | DIN 3977:1981-02 (gültig) | Tabelle 1 Meßstückdurchmesser, Abschnitt 6 zulässige Lage des Meßkreises |
| `ISO21771:2014` | DIN ISO 21771:2014-08 | V-Kreis, Grundteilung, Zahndicke am Grundzylinder, Erzeugungs-Profilverschiebung |
| `DIN3960:1987` | DIN 3960:1987-03 (zurückgezogen) | Gegenprobe; Form der Kugelwahl von STplus (Gl. (3.8.24)–(3.8.27)) |
| STplus 11.1F | Referenzläufe, Proben | Orakel für Zahlen; eigene Regeln für $k$ und $D_{\mathrm{M}}$ (`data/stplus_program`) |

In [ ]:
import math
from collections import Counter

import pandas as pd

import gearcore
from gearcore import generation as gn
from gearcore import inspection as ins
from gearcore import involute as iv
from gearcore import pair as pr
from gearcore.data import load_stplus, stplus_case_dirs, stplus_input_path
from gearcore.errors import InputRangeError
from gearcore.io.ste import load_ste, pair_input_from_ste
from gearcore.models.common import Pair
from gearcore.models.inputs import GearInput, PairInput, ToolProfile
from gearcore.parity import ParityVerdict, compare_inspection
from gearcore.stplus_program import (
    stplus_chord_diameter,
    stplus_inspection_choices,
    stplus_inspection_evidence,
    stplus_measuring_ball_diameter,
    stplus_measuring_ball_diameters,
    stplus_number_of_teeth_spanned,
    with_stplus_inspection_choices,
)
from gearcore.trace import equations_of

pd.set_option("display.precision", 6)
print(gearcore.__version__)

## 1 Was ein Prüfmaß einer Eingabe aussagt

**Normbezug.** DIN 21773:2014-08 §4, S. 8: Die Gleichungen liefern mit $x$ das **Nennmaß** und mit $x_{\mathrm{Es}}$, $x_{\mathrm{Em}}$, $x_{\mathrm{Ei}}$ das obere, mittlere und untere **Grenzmaß**. DIN ISO 21771 Gl. (123), (124), S. 67: $x_{\mathrm{E}} = x + E_{\mathrm{sn}} / (2\, m_{\mathrm{n}} \tan\alpha_{\mathrm{n}})$.

**Entscheidungen (ADR-115).** Ein Prüfmaß einer Eingabe sagt, welches Maß es ist (`DimensionKind`). Je Rad bestimmen zwei von {Profilverschiebung, Prüfmaß, Abmaße} das dritte; je Paar zwei von {$a_{\mathrm{w}}$, $x_1$, $x_2$}. Mit Achsabstand und den Prüfmaßen beider Räder ist die Summe der Abmaße bestimmt, nicht ihre Aufteilung: der Kern verlangt eine vierte Angabe und belegt nichts vor. Der `.ste`-Importer liest eine Zahnweite als oberes Grenzmaß, wie STplus sie liest.

Referenzpaar kst-B (vom Nutzer geprüft): Achsabstand und Zahnweiten beider Räder. STplus belegt das obere Zahndickenabmaß des Ritzels vor (Reihe c25: −85 µm); hier wird es ausdrücklich angegeben.

In [ ]:
imported = pair_input_from_ste(load_ste(stplus_input_path("kst_b")))
for note in imported.notes:
    if "upper limit" in note:
        print("Importer:", note)
try:
    pr.resolve_tooth_thickness(imported.pair)
except InputRangeError as error:
    print("ohne vierte Angabe:", error)

pinion = imported.pair.gears.pinion.model_copy(
    update={"tooth_thickness_allowance_um": (-85.0, -125.0)}
)
pair = imported.pair.model_copy(
    update={"gears": Pair(pinion=pinion, wheel=imported.pair.gears.wheel)}
)
resolved = pr.resolve_tooth_thickness(pair)
generation = gn.compute_generation(pair)
listing = load_stplus("kst_b", "geometry")
pd.DataFrame(
    {
        "Rad": ["Ritzel", "Rad"],
        "x": list(resolved.profile_shift_coefficient),
        "x STplus": listing["x"]["numbers"][:2],
        "E_sns in um": [allowance[0] for allowance in resolved.tooth_thickness_allowance_um],
        "A_sne STplus in mm": listing["A_sne"]["numbers"][:2],
        "x_E": [g.generating_profile_shift_coefficient for g in generation.gears.as_tuple()],
    }
)

## 2 Zahnweite

**Normbezug.** DIN 21773:2014-08 §7.2, Gl. (9), (12), (13), S. 12; Gl. (14)–(17), S. 13–14. Gegenprobe DIN 3960:1987-03 Gl. (3.8.13), (3.8.15), (3.8.18), S. 21.

$$k = \mathrm{INT}\left[\frac{z}{\pi}\left(\frac{\tan\alpha_{\mathrm{vt}}}{\cos^2\beta_{\mathrm{b}}} - \operatorname{inv}\alpha_{\mathrm{t}} - \frac{2\, x \tan\alpha_{\mathrm{n}}}{z}\right) + 1\right] \quad (9)$$

$$W_{\mathrm{k}} = m_{\mathrm{n}} \cos\alpha_{\mathrm{n}} \left[\pi\,(k - 0{,}5) + z \operatorname{inv}\alpha_{\mathrm{t}}\right] + 2\, x\, m_{\mathrm{n}} \sin\alpha_{\mathrm{n}} \quad (14)$$

Gl. (12) und (13) begrenzen $k$ auf den Bereich, in dem die Meßflächen zwischen Fuß- und Kopf-Formkreis berühren (dieselbe Klammer mit $\alpha_{\mathrm{Ff}}$ und $+1{,}5$ bzw. $\alpha_{\mathrm{Fa}}$ und $+0{,}5$).

**Befund.** Gl. (10) und die zweiten Formen von Gl. (12), (13) sind, wie gedruckt, innerhalb von INT um 0,5 kleiner als Gl. (9) und die ersten Formen ($s_{\mathrm{bn}}/p_{\mathrm{bn}}$ enthält die halbe Teilung). `gearcore` implementiert die ersten Formen; kst-B, Ritzel: Gl. (9) ergibt 5 wie STplus, Gl. (10) wie gedruckt 4 (`tests/test_inspection.py`, `test_eq_10_and_the_second_forms_of_eq_12_and_13_are_half_a_tooth_lower`).

In [ ]:
result = ins.compute_inspection(generation)
rows = []
for role, gear in zip(("Ritzel", "Rad"), result.gears.as_tuple(), strict=True):
    span = gear.span_measurement_mm
    rows.append(
        {
            "Rad": role,
            "k_min": gear.min_number_of_teeth_spanned,
            "k": gear.number_of_teeth_spanned,
            "k_max": gear.max_number_of_teeth_spanned,
            "W_k Nennmaß": span.nominal,
            "oberes": span.upper,
            "mittleres": span.mean,
            "unteres": span.lower,
            "Berührkreis": gear.span_measuring_circle_diameter_mm,
        }
    )
display(pd.DataFrame(rows))

for function in (ins.number_of_teeth_spanned, ins.span_measurement):
    print(function.__name__, "->", "; ".join(str(ref) for ref in equations_of(function)))

## 3 Kugel- und Rollenmaße

**Normbezug.** DIN 21773:2014-08 §8, Gl. (26), (27), S. 18 (Kugel, die am V-Zylinder berührt); Gl. (30)–(34), S. 20; §10 Gl. (35), (36), S. 21; §11 Gl. (37), S. 22. DIN 3977:1981-02 Tabelle 1, S. 1; Abschnitt 6, S. 2 (Meßkreis zwischen $-0{,}1\, m_{\mathrm{n}}$ und $+0{,}5\, m_{\mathrm{n}}$ um den V-Zylinder).

$$\operatorname{inv}\alpha_{\mathrm{Kt}} = \frac{D_{\mathrm{M}}}{z\, m_{\mathrm{n}} \cos\alpha_{\mathrm{n}}} - \eta + \operatorname{inv}\alpha_{\mathrm{t}} \quad (30) \qquad d_{\mathrm{K}} = \frac{d_{\mathrm{b}}}{\cos\alpha_{\mathrm{Kt}}} \quad (31)$$

$$M_{\mathrm{dK}} = d_{\mathrm{K}} + D_{\mathrm{M}} \quad (35,\ z\ \text{gerade}) \qquad M_{\mathrm{dK}} = d_{\mathrm{K}} \cos\frac{\pi}{2 z} + D_{\mathrm{M}} \quad (36,\ z\ \text{ungerade})$$

**Wahl von $D_{\mathrm{M}}$ (ADR-115).** Ohne Angabe: der nächstgrößere Wert der DIN 3977 Tabelle 1 über dem idealen Durchmesser der Gl. (26), (27), hochgesetzt, bis die Kugel über den Kopfzylinder ragt, auf der nutzbaren Flanke anliegt und den Fußkreis nicht berührt (mit Hinweis). Ein angegebener Durchmesser wird verwendet. Beispiel der Norm (DIN 3977 Bild 1): $z = 22$, $\beta = 30°$, $x = 0{,}5$ ergibt $D_{\mathrm{M}} = 1{,}85\, m_{\mathrm{n}}$; für $m_{\mathrm{n}} = 3{,}5$ mm ist 6,5 mm zu benutzen.

In [ ]:
# DIN 3977 Bild 1: z = 22, beta = 30 Grad, x = 0,5, m_n = 3,5 mm (Gegenrad und Werkzeug beliebig)
tool = ToolProfile(
    addendum_factor=1.25, tip_radius_factor=0.25, protuberance_mm=0.0, machining_allowance_mm=0.0
)
bild_1 = PairInput(
    normal_module_mm=3.5,
    normal_pressure_angle_deg=20.0,
    helix_angle_deg=30.0,
    gears=Pair(
        pinion=GearInput(
            number_of_teeth=22,
            profile_shift_coefficient=0.5,
            face_width_mm=40.0,
            tip_diameter_mm=99.412,
            tip_chamfer_radial_mm=0.0,
            tool=tool,
        ),
        wheel=GearInput(
            number_of_teeth=41,
            profile_shift_coefficient=0.0,
            face_width_mm=40.0,
            tip_diameter_mm=172.7,
            tip_chamfer_radial_mm=0.0,
            tool=tool,
        ),
    ),
)
example = ins.compute_inspection(gn.compute_generation(bild_1)).gears.pinion
print("idealer D_M:", example.ideal_measuring_ball_diameter_mm, "(Norm: 1,85 x 3,5 = 6,475)")
print("gewählt:", example.measuring_ball_diameter_mm, "mm (Norm: 6,5)")

rows = []
for role, gear in zip(("Ritzel", "Rad"), result.gears.as_tuple(), strict=True):
    balls = gear.diametral_two_ball_dimension_mm
    rows.append(
        {
            "Rad": role,
            "idealer D_M": gear.ideal_measuring_ball_diameter_mm,
            "D_M": gear.measuring_ball_diameter_mm,
            "d_K": gear.ball_centre_circle_diameter_mm,
            "Meßkreis d_M": gear.ball_measuring_circle_diameter_mm,
            "(d_M - d_v) / (2 m_n)": gear.measuring_circle_offset_factor,
            "M_dK Nennmaß": balls.nominal,
            "oberes": balls.upper,
            "unteres": balls.lower,
            "M_dZ": gear.diametral_two_roller_dimension_mm.nominal,
        }
    )
pd.DataFrame(rows)

## 4 Zahndickensehne

**Normbezug.** DIN 21773:2014-08 §5, Gl. (1)–(6), S. 9–10 (Sehne und Höhe über der Sehne am Teilzylinder und an einem Y-Zylinder); §6 Gl. (7), (8), S. 11 (konstante Sehne). Die Norm überläßt den Zylinder dem Anwender; STplus mißt auf der Mitte der Formkreise, $(d_{\mathrm{Ff}} + d_{\mathrm{Fa}})/2$ (Regel des Programms, `stplus_chord_diameter`).

In [ ]:
chords = Pair(
    pinion=stplus_chord_diameter(
        generation.gears.pinion.root_form_diameter_mm, generation.gears.pinion.tip_form_diameter_mm
    ),
    wheel=stplus_chord_diameter(
        generation.gears.wheel.root_form_diameter_mm, generation.gears.wheel.tip_form_diameter_mm
    ),
)
with_chord = ins.compute_inspection(generation, chord_diameter_mm=chords)
rows = []
for index, (role, gear) in enumerate(
    zip(("Ritzel", "Rad"), with_chord.gears.as_tuple(), strict=True)
):
    rows.append(
        {
            "Rad": role,
            "s_c (Teilzylinder)": gear.chordal_tooth_thickness_mm.nominal,
            "h_c": gear.height_above_chord_mm,
            "d_y": gear.y_diameter_mm,
            "s_cy": gear.chordal_tooth_thickness_at_y_mm.nominal,
            "s_cy STplus": listing["s_n-"]["numbers"][index],
            "h_cy": gear.height_above_chord_at_y_mm,
            "h_cy STplus": listing["h_a-"]["numbers"][index],
            "s_cc": gear.constant_chord_mm,
            "h_cc": gear.height_above_constant_chord_mm,
        }
    )
pd.DataFrame(rows)

## 5 Grenzmaße und Abmaßfaktoren

**Normbezug.** DIN 21773:2014-08 §14, Gl. (43)–(63), S. 24–26: $T_{\mathrm{sn}} = E_{\mathrm{sns}} - E_{\mathrm{sni}}$ (45), $E_{\mathrm{W}} = E_{\mathrm{sn}} \cos\alpha_{\mathrm{n}}$ (54), Grenzmaße aus dem mittleren Maß und dem Abmaßfaktor (z. B. Gl. (55), (56), (62), (63)).

`gearcore` rechnet die Grenzmaße nach §4 exakt mit $x_{\mathrm{Es}}$, $x_{\mathrm{Em}}$, $x_{\mathrm{Ei}}$. Die Abmaßfaktoren des §14 sind die Ableitungen: für die Zahnweite (linear in $x$) ist das dasselbe, für das Kugelmaß stimmt es bis auf die zweite Ordnung.

In [ ]:
gear = result.gears.pinion
T_sn = gear.tooth_thickness_tolerance_um
span, balls = gear.span_measurement_mm, gear.diametral_two_ball_dimension_mm
by_factor_span = ins.upper_limit_dimension(span.mean, T_sn, gear.span_allowance_factor)
by_factor_ball = ins.upper_limit_dimension(
    balls.mean, T_sn, gear.diametral_ball_dimension_allowance_factor
)
pd.DataFrame(
    {
        "Maß": ["Zahnweite", "Zweikugelmaß"],
        "oberes Grenzmaß nach §4": [span.upper, balls.upper],
        "nach §14 (Faktor)": [by_factor_span, by_factor_ball],
        "Unterschied in mm": [by_factor_span - span.upper, by_factor_ball - balls.upper],
    }
)

## 6 Vergleich mit STplus

STplus druckt die Prüfmaße der Nennverzahnung mit ihren Abmaßen. Der Vergleich (`parity.compare_inspection`) rechnet jede Zeile mit Fehlerfortpflanzung (gedruckte Stellen, einfache Genauigkeit von STplus). $k$ und $D_{\mathrm{M}}$ werden nicht dem Listing entnommen, sondern nach den Regeln von STplus aus den von `gearcore` erzeugten Formkreisen bestimmt und mit dem Listing verglichen (Abschnitt 7). Die Sehne wird auf dem von STplus gedruckten Zylinder verglichen, der Zylinder selbst im Rahmen der Genauigkeit der Formkreise von STplus.

Nicht verglichen, weil STplus sie anders definiert (`expected_differences.yaml`): der Abmaßfaktor des Kugelmaßes (STplus: Verhältnis der oberen Abmaße, Norm: Ableitung) und das Abmaß der Sehne.

In [ ]:
counts: dict[str, Counter] = {}
for folder in stplus_case_dirs():
    for row in compare_inspection(folder.name):
        counts.setdefault(row.field, Counter())[row.verdict] += 1
table = pd.DataFrame(
    {
        field: {verdict.value: counter[verdict] for verdict in ParityVerdict}
        for field, counter in counts.items()
    }
).T
display(table)
print(table.sum().to_dict())
assert table[ParityVerdict.DIFFERENT.value].sum() == 0

## 7 Was STplus selbst wählt: $k$ und $D_{\mathrm{M}}$

Das Handbuch sagt nicht, wie STplus $k$ und $D_{\mathrm{M}}$ wählt. Die Regeln wurden am 2026-10-04 experimentell gefunden (Umschaltpunkte über Kopfkreisdurchmesser und Profilverschiebung auf 0,0005 mm eingegrenzt, dann Zufallsräder) und liegen mit ihrem Beleg in `gearcore.stplus_program` (ADR-109: Regeln des Programms, nur auf Anforderung angewendet).

- **$k$:** $k = \min\left(\mathrm{INT}\left[(k_{\mathrm{min}} + k_{\mathrm{max}})/2\right] + 1,\ k_{\mathrm{max}}\right)$ mit $k_{\mathrm{min}}$, $k_{\mathrm{max}}$ nach Gl. (12), (13) beim oberen Abmaß. Die Norm wählt das $k$ am V-Zylinder (Gl. (9)).
- **$D_{\mathrm{M}}$:** eigene Tabelle (44 Werte, 1 bis 110 mm, nicht DIN 3977 Tabelle 1). A: der kleinste Wert, dessen $M_{\mathrm{dK}}$ beim oberen Abmaß über $d_{\mathrm{a}}$ liegt. B: zwei Tabellenwerte unter dem ersten Wert über $D_{\mathrm{T}}$, der Kugel, die nach der Form von DIN 3960 Gl. (3.8.24)–(3.8.27) auf der Mitte der Formkreise $T = (d_{\mathrm{Ff}} + d_{\mathrm{Fa}})/2$ berührt, mit $z_{\mathrm{n}} = z/\cos^3\beta$ für Faktor und Profilwinkel, aber $\operatorname{inv}\alpha_{\mathrm{t}}$ und $\eta$ des wirklichen Rads. $D_{\mathrm{M}} = \max(\mathrm{A}, \mathrm{B})$. Für Geradverzahnung ist $D_{\mathrm{T}}$ exakt, für Schrägverzahnung nicht.

Die Wahl ändert nur, welches Maß gedruckt wird, nicht das Rad. `gearcore` wählt nach der Norm; `with_stplus_inspection_choices` gibt die Wahl von STplus als Eingabe an, wenn die Maße eines STplus-Listings gebraucht werden.

In [ ]:
table_stplus = stplus_measuring_ball_diameters()
table_norm = ins.standard_measuring_ball_diameters()
print("nur DIN 3977 (ab 1 mm):", sorted({d for d in table_norm if d >= 1.0} - set(table_stplus)))
print("nur STplus:", sorted(set(table_stplus) - set(table_norm)))

# Beleg: 540 Zufallsräder aus STplus-Listings, alle folgen beiden Regeln
followed = Counter()
gears = [g for group in stplus_inspection_evidence()["groups"] for g in group["gears"]]
for g in gears:
    alpha, beta = math.radians(g["alpha_n_deg"]), math.radians(g["beta_deg"])
    z, m_n, x_E = int(g["z"]), g["m_n"], g["x_E"]
    d_Ff = max(g["d_Ff"], iv.base_diameter(z, m_n, alpha, beta))
    k = stplus_number_of_teeth_spanned(
        ins.min_number_of_teeth_spanned(z, m_n, x_E, alpha, beta, d_Ff),
        ins.max_number_of_teeth_spanned(z, m_n, x_E, alpha, beta, g["d_Fa"]),
    )
    ball = stplus_measuring_ball_diameter(z, m_n, x_E, alpha, beta, g["d_a"], g["d_Ff"], g["d_Fa"])
    followed["k"] += k == int(g["k"])
    followed["D_M"] += ball == g["D_M"]
print(len(gears), "Räder:", dict(followed))

# kst-B: die Eingabe nennt k = 5 / 8; STplus druckt das Rad über sein eigenes k
print("Wahl von STplus (k, D_M):", stplus_inspection_choices(generation))
as_stplus = ins.compute_inspection(with_stplus_inspection_choices(generation))
pd.DataFrame(
    {
        "Rad": ["Ritzel", "Rad"],
        "k Norm/Eingabe": [g.number_of_teeth_spanned for g in result.gears.as_tuple()],
        "W_k": [g.span_measurement_mm.nominal for g in result.gears.as_tuple()],
        "k STplus": [g.number_of_teeth_spanned for g in as_stplus.gears.as_tuple()],
        "W_k dazu": [g.span_measurement_mm.nominal for g in as_stplus.gears.as_tuple()],
        "W_k Listing": listing["W_k"]["numbers"][:2],
        "x unverändert": [
            a.profile_shift_coefficient == b.profile_shift_coefficient
            for a, b in zip(result.gears.as_tuple(), as_stplus.gears.as_tuple(), strict=True)
        ],
    }
)

## 8 Grenzen dieses Inkrements und Erweiterungspunkte

- Nur Außenverzahnung; Hohlräder (DIN 21773 §9, DIN 3977 Abschnitt 6 für Innenverzahnung) sind ein Erweiterungspunkt.
- Zahndickensehne und $x_{\mathrm{F}}$ (`PROFILVERSCHIEBUNG_F`) als **Eingabe** werden nicht übersetzt (`NotSupportedError`); als Ausgabe ist die Sehne vorhanden.
- Dateien, die nur Achsabstand und Prüfmaße geben (kst-A/B/C), brauchen die Vorbelegung der Abmaßreihe von STplus (DIN 3967, Inkrement 5); bis dahin verlangt der Kern die vierte Angabe.
- Zweiflanken-Wälzprüfung (DIN 21773 §12) und DIN ISO 1328-2: Inkrement 5.
- Modifizierte Flanken (Kopf-/Fußrücknahme) verschieben den nutzbaren Bereich von $k$ und $D_{\mathrm{M}}$; `gearcore` rechnet mit den Formkreisen der unmodifizierten Flanke.

Einzelheiten: `known_limits.md` (INS-01 ff.), `extension_points.md`, `norm_map.md` (Befunde aus Inkrement 4).